# Mini-Projeto Avaliativo - Módulo 2

### Importação das bibliotecas

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.neighbors import KNeighborsClassifier
import time

from sklearn.ensemble import RandomForestClassifier

from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from sklearn.datasets import fetch_openml

## Fase 1: Carregamento e Análise Exploratória de Imagens (EDA)

#### 1.1 Importação e carregamento do MNIST

In [ ]:
print("Baixando o dataset MNIST...")
mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
print("Finalizou o processo de baixar o dataset MNIST!")


#### 1.2 Separação entre features e target

In [ ]:
X = mnist.data
y = mnist.target.astype(int)

print("Features (X):", X.shape)
print("Target (y):", y.shape)

### 1.3 Dimensionalidade dos dados

In [ ]:
print("Dimensionalidade dos dados:")
print(f"X: {X.shape}")
print(f"y: {y.shape}")

print("\nQuantidade de imagens:", X.shape[0])
print("Quantidade de features por imagem:", X.shape[1])

#### 1.4 Distribuição das classes

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()

print("Distribuição das classes:")
print(class_counts)

In [ ]:
plt.figure(figsize=(8, 5))

class_counts.plot(kind='bar')

plt.title('Distribuição das Classes no MNIST')
plt.xlabel('Dígito')
plt.ylabel('Quantidade de imagens')
plt.xticks(rotation=0)

plt.show()

#### 1.5 Visualização dos dígitos

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))

for digit, ax in enumerate(axes.ravel()):
    # Localiza a primeira imagem correspondente ao dígito
    index = np.where(y == digit)[0][0]
    
    # Reconstrói o vetor de 784 pixels para 28x28
    image = X[index].reshape(28, 28)
    
    ax.imshow(image, cmap='gray')
    ax.set_title(f'Dígito: {digit}')
    ax.axis('off')

plt.tight_layout()
plt.show()

#### 1.6 Estrutura dos pixels

In [ ]:
print("Valor mínimo dos pixels:", X.min())
print("Valor máximo dos pixels:", X.max())

In [ ]:
sample_index = 0

image = X[sample_index].reshape(28, 28)

print("Dimensão da imagem:", image.shape)

print("\nMatriz de pixels:")
print(image)

In [ ]:
plt.figure(figsize=(5, 5))

plt.imshow(image, cmap='gray')

plt.title(f'Dígito: {y[sample_index]}')
plt.axis('off')

plt.show()

#### 1.7 Interpretação dos dados

O dataset MNIST é composto por imagens de dígitos manuscritos com resolução de **28 × 28 pixels**. Cada pixel possui uma intensidade que varia de **0 a 255**, onde valores próximos de 0 representam regiões mais escuras e valores próximos de 255 representam regiões mais claras.

Embora a imagem possua originalmente duas dimensões (28 × 28), os modelos de Machine Learning utilizados neste projeto recebem os dados em formato vetorial. Dessa forma, cada imagem é transformada em um vetor de **784 features (28 × 28 = 784)**.

A matriz `X` contém as imagens vetorizadas, enquanto o vetor `y` contém o rótulo correspondente a cada imagem, representando um dos dez dígitos possíveis, de **0 a 9**.

A análise da distribuição das classes nos permite verificar se existe algum desequilíbrio relevante entre os diferentes dígitos antes da etapa de treinamento dos modelos. 

Após análise, conseguimos ver que não há desequilíbrio relevante entre os diferentes dígitos.


## Fase 2: Pipeline de Pré-processamento e Divisão dos Dados

#### 2.1 Divisão estratificada dos dados

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [ ]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=2/3,
    random_state=42,
    stratify=y_temp
)

### 2.2 Verificação das dimensões

In [ ]:
print("Dimensões dos conjuntos:")

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")

print(f"\nX_val: {X_val.shape}")
print(f"y_val: {y_val.shape}")

print(f"\nX_test: {X_test.shape}")
print(f"y_test: {y_test.shape}")

### 2.3 Verificação da estratificação

In [ ]:
print("Distribuição percentual das classes:\n")

print("Treino:")
print(
    pd.Series(y_train)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nValidação:")
print(
    pd.Series(y_val)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

print("\nTeste:")
print(
    pd.Series(y_test)
    .value_counts(normalize=True)
    .sort_index()
    .round(4)
)

### 2.4 Normalização dos pixels

In [ ]:
X_train = X_train.astype('float32') / 255.0
X_val = X_val.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

### 2.5 Verificação da normalização

In [ ]:
print("Após a normalização:")

print(f"Valor mínimo em X_train: {X_train.min()}")
print(f"Valor máximo em X_train: {X_train.max()}")

print(f"\nValor mínimo em X_val: {X_val.min()}")
print(f"Valor máximo em X_val: {X_val.max()}")

print(f"\nValor mínimo em X_test: {X_test.min()}")
print(f"Valor máximo em X_test: {X_test.max()}")

### 2.6 Visualização de imagem após normalização

In [ ]:
sample_index = 0

plt.figure(figsize=(5, 5))

plt.imshow(
    X_train[sample_index].reshape(28, 28),
    cmap='gray'
)

plt.title(f'Dígito: {y_train[sample_index]}')
plt.axis('off')

plt.show()

### 2.7 Análise

Após o pré-processamento, o dataset foi dividido de forma estratificada em treino, validação e teste, mantendo a proporção das dez classes entre os conjuntos.

Os pixels foram normalizados de uma escala original de 0–255 para 0.0–1.0, preparando os dados para o treinamento dos modelos. A normalização coloca todas as características em uma escala comum, facilitando o processo de otimização e contribuindo para a convergência de modelos sensíveis à escala dos dados.

Essa transformação também é especialmente importante para algoritmos baseados em distância, como o KNN, pois evita que diferenças de escala prejudiquem o cálculo das distâncias entre as observações.

O conjunto de teste permanece separado para a avaliação final dos modelos, evitando que suas informações sejam utilizadas durante o processo de desenvolvimento e ajuste.

## Fase 3: Implementação e Treinamento dos 3 Modelos

### 3.1 KNN

#### Treinamento e avaliação das configurações

In [ ]:

knn_configs = [
    {"n_neighbors": 3, "weights": "uniform"},
    {"n_neighbors": 5, "weights": "uniform"},
    {"n_neighbors": 7, "weights": "distance"},
]

knn_results = []

for config in knn_configs:
    model = KNeighborsClassifier(
        n_neighbors=config["n_neighbors"],
        weights=config["weights"]
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    knn_results.append({
        "n_neighbors": config["n_neighbors"],
        "weights": config["weights"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

knn_results_df = pd.DataFrame(knn_results)
knn_results_df

#### Escolha do melhor KNN

In [ ]:
best_knn_config = knn_results_df.loc[
    knn_results_df["val_accuracy"].idxmax()
]

best_knn_config

#### Treinamento do modelo final

In [ ]:
best_knn = KNeighborsClassifier(
    n_neighbors=int(best_knn_config["n_neighbors"]),
    weights=best_knn_config["weights"]
)

start_time = time.time()
best_knn.fit(X_train, y_train)
knn_training_time = time.time() - start_time

print(f"Tempo de treinamento: {knn_training_time:.2f} segundos")

### 3.2 Random Forest

#### Treinamento e avaliação das configurações

In [ ]:
rf_configs = [
    {"n_estimators": 100, "max_depth": 15},
    {"n_estimators": 150, "max_depth": 20},
    {"n_estimators": 200, "max_depth": None},
]

rf_results = []

for config in rf_configs:
    model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        random_state=42,
        n_jobs=-1
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    rf_results.append({
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

rf_results_df = pd.DataFrame(rf_results)
rf_results_df

#### Escolha da melhor configuração

In [ ]:
best_rf_config = rf_results_df.loc[
    rf_results_df["val_accuracy"].idxmax()
]

best_rf_config

#### Treinamento do modelo final

In [ ]:
best_rf = RandomForestClassifier(
    n_estimators=int(best_rf_config["n_estimators"]),
    max_depth=(
        int(best_rf_config["max_depth"])
        if pd.notna(best_rf_config["max_depth"])
        else None
    ),
    random_state=42,
    n_jobs=-1
)

start_time = time.time()
best_rf.fit(X_train, y_train)
rf_training_time = time.time() - start_time

print(f"Tempo de treinamento: {rf_training_time:.2f} segundos")

### 3.3 MLP

#### Treinamento e avaliação das configurações

In [ ]:
mlp_configs = [
    {"hidden_layer_sizes": (64,), "alpha": 0.0001},
    {"hidden_layer_sizes": (128,), "alpha": 0.0001},
    {"hidden_layer_sizes": (128,), "alpha": 0.001},
]

mlp_results = []

for config in mlp_configs:
    model = MLPClassifier(
        hidden_layer_sizes=config["hidden_layer_sizes"],
        alpha=config["alpha"],
        max_iter=50,
        random_state=42,
        early_stopping=True
    )

    start_time = time.time()
    model.fit(X_train, y_train)
    training_time = time.time() - start_time

    train_pred = model.predict(X_train)
    val_pred = model.predict(X_val)

    train_accuracy = accuracy_score(y_train, train_pred)
    val_accuracy = accuracy_score(y_val, val_pred)

    mlp_results.append({
        "hidden_layer_sizes": config["hidden_layer_sizes"],
        "alpha": config["alpha"],
        "train_accuracy": train_accuracy,
        "val_accuracy": val_accuracy,
        "training_time": training_time
    })

mlp_results_df = pd.DataFrame(mlp_results)
mlp_results_df

#### Escolha da melhor configuração

In [ ]:
best_mlp_config = mlp_results_df.loc[
    mlp_results_df["val_accuracy"].idxmax()
]

best_mlp_config

#### Treinamento do modelo final

In [ ]:
best_mlp = MLPClassifier(
    hidden_layer_sizes=best_mlp_config["hidden_layer_sizes"],
    alpha=best_mlp_config["alpha"],
    max_iter=20,
    random_state=42,
    early_stopping=True
)

start_time = time.time()
best_mlp.fit(X_train, y_train)
mlp_training_time = time.time() - start_time

print(f"Tempo de treinamento: {mlp_training_time:.2f} segundos")

### 3.4 Comparação dos modelos

#### Comparação de desempenho e tempo de treinamento

In [ ]:
model_comparison = pd.DataFrame({
    "Modelo": ["KNN", "Random Forest", "MLP"],
    "Accuracy Validação": [
        best_knn_config["val_accuracy"],
        best_rf_config["val_accuracy"],
        best_mlp_config["val_accuracy"]
    ],
    "Tempo Treinamento (s)": [
        knn_training_time,
        rf_training_time,
        mlp_training_time
    ]
})

model_comparison.sort_values(
    "Accuracy Validação",
    ascending=False
)

### 3.5 Análise de overfitting

#### Comparação entre treino e validação

In [ ]:
overfitting_comparison = pd.DataFrame({
    "Modelo": ["KNN", "Random Forest", "MLP"],
    "Accuracy Treino": [
        best_knn_config["train_accuracy"],
        best_rf_config["train_accuracy"],
        best_mlp_config["train_accuracy"]
    ],
    "Accuracy Validação": [
        best_knn_config["val_accuracy"],
        best_rf_config["val_accuracy"],
        best_mlp_config["val_accuracy"]
    ]
})

overfitting_comparison["Diferença"] = (
    overfitting_comparison["Accuracy Treino"]
    - overfitting_comparison["Accuracy Validação"]
)

overfitting_comparison

#### Análise

Após o treinamento e ajuste dos três modelos, o **MLP apresentou o melhor resultado no conjunto de validação, com 97,69% de acurácia**, seguido pelo KNN com 97,11% e pelo Random Forest com 96,64%.

O KNN apresentou a menor diferença entre as acurácias de treinamento e validação, indicando boa capacidade de generalização e baixo indício de overfitting. O MLP apresentou uma diferença moderada de aproximadamente 2,13 pontos percentuais, mantendo, porém, a melhor acurácia de validação. Já o Random Forest apresentou a maior diferença, de 3,31 pontos percentuais, indicando maior tendência ao overfitting.

Considerando a acurácia de validação como principal critério de seleção, o **MLP foi escolhido como o modelo de melhor desempenho nesta etapa**. Entretanto, os três modelos serão avaliados posteriormente no conjunto de teste independente para verificar seu desempenho final e capacidade de generalização.


## Fase 4: Avaliação Comparativa de Desempenho

### 4.1 Preparação das previsões

#### Previsões no conjunto de teste

In [ ]:
# Previsões dos modelos no conjunto de teste
y_pred_knn = best_knn.predict(X_test)
y_pred_rf = best_rf.predict(X_test)
y_pred_mlp = best_mlp.predict(X_test)

### 4.2 Métricas de avaliação

#### Cálculo das métricas

In [ ]:
models = {
    "KNN": y_pred_knn,
    "Random Forest": y_pred_rf,
    "MLP": y_pred_mlp
}

test_results = []

for model_name, y_pred in models.items():
    test_results.append({
        "Modelo": model_name,
        "Acurácia": accuracy_score(y_test, y_pred),
        "Precisão (ponderada)": precision_score(
            y_test, y_pred, average="weighted"
        ),
        "Revocação (ponderada)": recall_score(
            y_test, y_pred, average="weighted"
        ),
        "F1-score (ponderado)": f1_score(
            y_test, y_pred, average="weighted"
        )
    })

test_results_df = pd.DataFrame(test_results)

test_results_df

#### Matriz de confusão — KNN

In [ ]:
cm_knn = confusion_matrix(y_test, y_pred_knn)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_knn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(10),
    yticklabels=range(10)
)

plt.xlabel("Dígito previsto")
plt.ylabel("Dígito verdadeiro")
plt.title("Matriz de Confusão — KNN")
plt.tight_layout()
plt.show()

#### Matriz de confusão — Random Forest

In [ ]:
cm_rf = confusion_matrix(y_test, y_pred_rf)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_rf,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(10),
    yticklabels=range(10)
)

plt.xlabel("Dígito previsto")
plt.ylabel("Dígito verdadeiro")
plt.title("Matriz de Confusão — Random Forest")
plt.tight_layout()
plt.show()

#### Matriz de confusão — MLP

In [ ]:
cm_mlp = confusion_matrix(y_test, y_pred_mlp)

plt.figure(figsize=(10, 8))
sns.heatmap(
    cm_mlp,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=range(10),
    yticklabels=range(10)
)

plt.xlabel("Dígito previsto")
plt.ylabel("Dígito verdadeiro")
plt.title("Matriz de Confusão — MLP")
plt.tight_layout()
plt.show()

### 4.4 Identificação das maiores confusões

In [ ]:
def most_confused_pair(cm):
    cm_off_diagonal = cm.copy()
    np.fill_diagonal(cm_off_diagonal, 0)

    true_digit, predicted_digit = np.unravel_index(
        np.argmax(cm_off_diagonal),
        cm_off_diagonal.shape
    )

    return true_digit, predicted_digit, cm_off_diagonal[
        true_digit, predicted_digit
    ]


confusion_results = []

for model_name, cm in {
    "KNN": cm_knn,
    "Random Forest": cm_rf,
    "MLP": cm_mlp
}.items():

    true_digit, predicted_digit, count = most_confused_pair(cm)

    confusion_results.append({
        "Modelo": model_name,
        "Dígito verdadeiro": true_digit,
        "Dígito previsto": predicted_digit,
        "Quantidade de erros": count
    })

confusion_results_df = pd.DataFrame(confusion_results)

confusion_results_df

### 4.5 Comparação final

#### Comparação de desempenho e custo computacional

In [ ]:
training_times = {
    "KNN": knn_training_time,
    "Random Forest": rf_training_time,
    "MLP": mlp_training_time
}

test_results_df["Tempo de treinamento (s)"] = (
    test_results_df["Modelo"].map(training_times)
)

test_results_df

### 4.6 Seleção do melhor modelo

In [ ]:
best_model_row = test_results_df.loc[
    test_results_df["Acurácia"].idxmax()
]

best_model_name = best_model_row["Modelo"]

print(f"O melhor modelo é: {best_model_name}")

### 4.7 Análise

No conjunto de teste independente, o **MLP apresentou o melhor desempenho, com 97,48% de acurácia**, seguido pelo KNN, com 96,93%, e pelo Random Forest, com 96,49%. As métricas de precisão, revocação e F1-score ponderados apresentaram valores muito próximos da acurácia em todos os modelos, indicando desempenho consistente entre as classes.

Nas matrizes de confusão, o par de dígitos que apresentou maior confusão foi **4 e 9**. No KNN, a maior confusão ocorreu de 4 para 9, com 38 casos, enquanto no Random Forest foram 32 casos. No MLP, a maior confusão ocorreu no sentido contrário, de 9 para 4, com 17 casos. Esses resultados indicam que características visuais semelhantes entre esses dígitos podem dificultar sua distinção.

Em relação ao custo computacional de treinamento, o KNN apresentou o menor tempo, com aproximadamente 0,26 segundo, enquanto o Random Forest e o MLP exigiram aproximadamente 23,65 e 28,09 segundos, respectivamente. Apesar do maior custo de treinamento, o **MLP apresentou a maior acurácia no conjunto de teste e foi selecionado como o melhor modelo para a próxima etapa**.


### Fase 5.1 - Desafio (A) Treinamento Restrito com Classes Ocultadas (Class Masking)

#### Definição das classes ocultadas

In [ ]:
hidden_classes = [8, 9]

print("Classes ocultadas:", hidden_classes)

#### Preparação dos dados de treinamento

In [ ]:
known_mask = ~np.isin(y_train, hidden_classes)

X_train_known = X_train[known_mask]
y_train_known = y_train[known_mask]

hidden_mask_test = np.isin(y_test, hidden_classes)

X_test_hidden = X_test[hidden_mask_test]
y_test_hidden = y_test[hidden_mask_test]

print("Classes presentes no treinamento:", np.unique(y_train_known))
print("Classes ocultadas:", hidden_classes)
print("Amostras de treinamento:", X_train_known.shape)
print("Amostras para teste OOD:", X_test_hidden.shape)

#### Treinamento do modelo com classes restritas

In [ ]:
mlp_hidden = MLPClassifier(
    hidden_layer_sizes=(128,),
    alpha=0.001,
    max_iter=50,
    random_state=42,
    early_stopping=True
)

mlp_hidden.fit(X_train_known, y_train_known)

### Fase 5.2 - Desafio (B) Teste de Generalização Extrema (Inferência OOD)

#### Inferência sobre as classes ocultadas

In [ ]:
y_pred_hidden = mlp_hidden.predict(X_test_hidden)

print("Classes verdadeiras:", np.unique(y_test_hidden))
print("Classes previstas:", np.unique(y_pred_hidden))

#### Matriz de confusão das classes ocultadas

In [ ]:
hidden_confusion = pd.crosstab(
    pd.Series(y_test_hidden, name="Dígito verdadeiro"),
    pd.Series(y_pred_hidden, name="Dígito previsto")
)

hidden_confusion = hidden_confusion.reindex(
    index=hidden_classes,
    columns=range(8),
    fill_value=0
)

plt.figure(figsize=(10, 4))

sns.heatmap(
    hidden_confusion,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.xlabel("Dígito previsto")
plt.ylabel("Dígito verdadeiro")
plt.title("Matriz de Confusão — Classes Ocultadas")
plt.tight_layout()
plt.show()

#### Distribuição das previsões

In [ ]:
hidden_prediction_counts = pd.crosstab(
    pd.Series(y_test_hidden, name="Dígito verdadeiro"),
    pd.Series(y_pred_hidden, name="Dígito previsto")
)

hidden_prediction_counts

#### Confiança das previsões

In [ ]:
hidden_probabilities = mlp_hidden.predict_proba(X_test_hidden)

max_probabilities = hidden_probabilities.max(axis=1)

print(f"Confiança média: {max_probabilities.mean():.4f}")
print(f"Confiança mínima: {max_probabilities.min():.4f}")
print(f"Confiança máxima: {max_probabilities.max():.4f}")

#### Distribuição da confiança das previsões

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(max_probabilities, bins=20)

plt.xlabel("Maior probabilidade prevista")
plt.ylabel("Quantidade de imagens")
plt.title("Confiança do modelo nas classes ocultadas")
plt.tight_layout()
plt.show()

### Análise

O modelo apresentou previsões para as classes 8 e 9 mesmo sem ter recebido exemplos dessas classes durante o treinamento. As imagens foram associadas às classes conhecidas, demonstrando a limitação de um classificador fechado diante de dados fora da distribuição (OOD). A análise das probabilidades também permite verificar se o modelo mantém elevada confiança nessas previsões, caracterizando um possível comportamento de overconfidence.

O teste com as classes ocultadas demonstrou que o modelo não consegue reconhecer diretamente classes que não foram apresentadas durante o treinamento. Mesmo recebendo imagens pertencentes às classes ocultadas, o MLP foi obrigado a classificá-las entre as classes conhecidas.

Esse comportamento evidencia uma limitação importante de modelos de classificação: quando recebem uma entrada de uma classe desconhecida, eles podem produzir uma previsão entre as classes disponíveis, mesmo quando nenhuma delas representa corretamente a entrada.

A análise das probabilidades também permite observar o fenômeno de **overconfidence**, no qual o modelo pode apresentar alta confiança em uma previsão incorreta. Dessa forma, uma probabilidade elevada não deve ser interpretada como garantia de que a classificação está correta, principalmente quando o modelo recebe dados fora das classes utilizadas no treinamento.


### Fase 5.3 - Desafio (C) Inferência com Imagens Manuscritas Próprias

#### Carregamento das imagens manuscritas

In [ ]:
from PIL import Image
from pathlib import Path

image_paths = sorted(Path("nums").glob("*.png"))

print(f"Quantidade de imagens encontradas: {len(image_paths)}")

for path in image_paths:
    print(path)

#### Visualização das imagens originais

In [ ]:
fig, axes = plt.subplots(
    1,
    len(image_paths),
    figsize=(3 * len(image_paths), 3)
)

if len(image_paths) == 1:
    axes = [axes]

for ax, path in zip(axes, image_paths):
    image = Image.open(path)

    ax.imshow(image, cmap="gray")
    ax.set_title(path.stem)
    ax.axis("off")

plt.tight_layout()
plt.show()

#### Pré-processamento das imagens

In [ ]:
def preprocess_image(path):
    image = Image.open(path).convert("L")

    image_array = np.array(image, dtype="float32") / 255.0
    image_vector = image_array.reshape(1, -1)

    return image_array, image_vector

In [ ]:
processed_images = []
processed_vectors = []

for path in image_paths:
    image_array, image_vector = preprocess_image(path)

    processed_images.append(image_array)
    processed_vectors.append(image_vector)

X_own = np.vstack(processed_vectors)

print("Formato das imagens processadas:", X_own.shape)
print("Valor mínimo:", X_own.min())
print("Valor máximo:", X_own.max())

#### Inferência com o melhor modelo

In [ ]:
own_predictions = best_mlp.predict(X_own)

print("Predições:")
for path, prediction in zip(image_paths, own_predictions):
    print(f"{path.stem}: {prediction}")

#### Probabilidades das previsões

In [ ]:
own_probabilities = best_mlp.predict_proba(X_own)

for path, prediction, probabilities in zip(
    image_paths,
    own_predictions,
    own_probabilities
):
    print(f"\nImagem: {path.stem}")
    print(f"Predição: {prediction}")
    print(f"Probabilidade: {probabilities[prediction]:.4f}")

#### Distribuição das probabilidades por imagem

In [ ]:
n_images = len(image_paths)
n_cols = 5
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, path, probabilities, prediction in zip(
    axes,
    image_paths,
    own_probabilities,
    own_predictions
):
    ax.bar(range(10), probabilities)
    ax.set_xticks(range(10))
    ax.set_xlabel("Dígito")
    ax.set_ylabel("Probabilidade")
    ax.set_title(
        f"Imagem: {path.stem} | Predição: {prediction}"
    )
    ax.set_ylim(0, 1)

# Remoção de eixos não utilizados
for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Conclusão — Imagens manuscritas próprias

Nas dez imagens manuscritas próprias, correspondentes aos dígitos de 0 a 9, o modelo classificou corretamente todas as imagens, alcançando 100% de acurácia neste conjunto.

Apesar dos acertos, houve diferenças na confiança das previsões. Os dígitos 0, 2, 3, 4, 5, 8 e 9 apresentaram probabilidades superiores a 98%, enquanto os dígitos 1 e 7 apresentaram probabilidades de 92,06% e 95,72%, respectivamente. O dígito 6 apresentou a menor confiança, com 69,29%, embora também tenha sido classificado corretamente.

Durante o experimento, observou-se que o tamanho e a ocupação dos dígitos dentro da imagem podem influenciar a confiança das previsões. Isso ocorre porque imagens manuscritas próprias podem apresentar diferenças de escala, posicionamento e formato em relação às imagens utilizadas no treinamento do MNIST.

O experimento demonstra que o modelo apresentou boa capacidade de generalização para as imagens produzidas, mas também evidencia que a confiança das previsões pode variar de acordo com as características da entrada.


#### Teste com imagens manuscritas com maior variação

In [ ]:
# Carregamento das imagens da pasta num2
image_paths_num2 = sorted(Path("nums2").glob("*.png"))

print(f"Quantidade de imagens encontradas: {len(image_paths_num2)}")

for path in image_paths_num2:
    print(path)

In [ ]:
# Pré-processamento das imagens
processed_vectors_num2 = []

for path in image_paths_num2:
    _, image_vector = preprocess_image(path)
    processed_vectors_num2.append(image_vector)

X_num2 = np.vstack(processed_vectors_num2)

print("Formato das imagens processadas:", X_num2.shape)
print("Valor mínimo:", X_num2.min())
print("Valor máximo:", X_num2.max())

In [ ]:
# Previsões
predictions_num2 = best_mlp.predict(X_num2)
probabilities_num2 = best_mlp.predict_proba(X_num2)

print("Predições:")

for path, prediction, probabilities in zip(
    image_paths_num2,
    predictions_num2,
    probabilities_num2
):
    print(f"\nImagem: {path.stem}")
    print(f"Predição: {prediction}")
    print(f"Probabilidade: {probabilities[prediction]:.4f}")

In [ ]:
# Visualização das imagens
n_images = len(image_paths_num2)
n_cols = 5
n_rows = int(np.ceil(n_images / n_cols))

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(4 * n_cols, 4 * n_rows)
)

axes = np.array(axes).reshape(-1)

for ax, image, path, prediction, probabilities in zip(
    axes,
    [preprocess_image(path)[0] for path in image_paths_num2],
    image_paths_num2,
    predictions_num2,
    probabilities_num2
):
    ax.imshow(image, cmap="gray")
    ax.set_title(
        f"Real: {path.stem} | Pred: {prediction}\n"
        f"Confiança: {probabilities[prediction]:.1%}"
    )
    ax.axis("off")

for ax in axes[n_images:]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Conclusão — Imagens com maior variação

No segundo experimento, utilizando imagens manuscritas com maior variação de tamanho e formato, o modelo classificou corretamente 4 das 10 imagens, correspondendo a uma acurácia de 40%.

O resultado demonstra uma redução significativa de desempenho em relação ao primeiro conjunto, evidenciando a influência das características das imagens na capacidade de generalização do modelo.

Também foram observados casos de **overconfidence**. O dígito 8 foi classificado como 3 com 98,53% de probabilidade, enquanto o dígito 9 foi classificado como 3 com 99,77% de probabilidade. Esses casos mostram que o modelo pode apresentar alta confiança mesmo quando sua previsão está incorreta.

Por outro lado, o dígito 4 foi classificado como 7 com apenas 46,16% de probabilidade, demonstrando que o modelo também pode apresentar menor confiança quando a entrada é mais ambígua.
